# 🫀 퀘스트 46 · Q4-B — **주 관문을 고친다** (burden 특징 재판정)

| | **MedKOS / `notebooks/quest46_q4b_burden_feature_v2.ipynb`** |
|---|---|
| 퀘스트 | `ailab-2026-0046` — 층② 점수 눈금 |
| 부모 런 | `quest46_q4_burden_feature`(`20260804T2349`) · `quest46_q3b_prior_shuffle` |
| 성격 | **주 관문 교체 + 검정력 확보** — 1차는 잘못된 팔을 주 관문에 뒀다 |

## 1차가 무엇을 틀렸나

```
전역 PR-AUC     raw 0.3362 · A_oracle 0.5514 · **B_add 0.5725** · B_int 0.4116
주 관문 D2      `B_int − A_oracle` = −0.1398  → 「B 가 A 를 못 이긴다」
그런데 D2b      `B_add − A_oracle` = **+0.0212**   ← 구조 대조로 밀어둔 팔이 최고였다
```

**왜 그렇게 됐나 — 내 이론이 실측에 반증됐다.**

| 명제 | 실측 | 판정 |
|---|---|---|
| burden **기여분**이 레코드별 상수다 | 레코드 내 SD **4.46e-16** | ✅ **참** |
| 따라서 모델 전체가 A 와 같은 구조다 | raw 대비 레코드 내 ρ **0.792062** · 매크로 **+0.0398** | ❌ **거짓** |

참인 명제에서 거짓인 결론을 끌어냈다. **특징을 추가하면 로지스틱이 리듬 계수를 전부 다시
적합한다.** burden 항이 상수여도 나머지 가중치가 재조정되어 **레코드 내 판별이 바뀐다**.
합성(ρ 0.99996)에서는 안 보였고 실제 데이터(ρ 0.792)에서 크게 나타났다 — 합성에
burden 의존 결정경계가 없었기 때문이고, **그래서 픽스처가 못 잡았다**.

## ★★★ 1차의 진짜 발견 — 매크로가 올랐다

```
B_add_oracle  매크로 Δ vs raw  +0.039773 [+0.005381, +0.074179]
B_int_oracle                   +0.045094 [+0.014269, +0.074563]
B_int_em                       +0.037272 [+0.007368, +0.067788]
```

셋 다 CI 가 0 을 뗀다 — **비열등이 아니라 우월**이다. 그리고 이건 **A 계열이 원리적으로
못 하는 일**이다: A 는 레코드별 상수 시프트라 매크로가 **정의상 불변**이고, 1차에서
실제로 `A_oracle` 매크로 = `raw` 매크로 = 0.5389 로 완전히 같았다.

R11 이 매크로를 주 지표로 삼으라고 한 걸 생각하면 이게 1차의 가장 중요한 수치인데,
사전등록은 매크로를 **비열등 관문**으로만 뒀다.

## 무엇을 바꾸나

1. **주 관문을 `B_add − A` 로** — 사전등록 문구(「burden 을 **별도 특징**으로」)에 충실하게.
   `B_int` 는 **과적합 진단**으로 강등한다(1차 실측 0.4116 ≪ B_add 0.5725).
2. **매크로를 공동 주 지표로 승격** — 검정력이 실제로 있는 유일한 대비이고, A 가 못 하는
   일이며, R11 이 원래 요구한 지표다. 전역은 공동 주 지표로 함께 읽는다.
3. **`B_add_em` 팔 추가** — 1차에 **없었다**. 배포 가능한 최선 후보인데 빠져 있었다.
4. **LORO 로 전환** — 1차는 TEST 20 레코드뿐이라 전역 대비 MDE 가 0.1614 였다.
   LORO 면 **56 레코드 전부**가 평가에 들어간다.
5. **E1 에서 두 명제를 분리해 보고** — 「기여분은 상수」와 「모델 전체가 A 와 같다」는
   다른 말이고, 후자는 1차에서 **반증됐다**.

## 관문 (재사전등록)

| 관문 | 무엇 | 통과 기준 |
|---|---|---|
| **E0** | 코호트·항등 — A 팔의 매크로가 raw 와 **정확히 같은가** | 구성 항등(상수 시프트). 깨지면 **중단** |
| **E1 ★★ 구조** | burden 기여분(상수) **그리고** 모델 전체(재적합) | 관문 아님. **두 명제를 분리**해 보고 |
| **E2 ★★★ 공동 주 관문(매크로)** | `B_add − A` 매크로 짝지은 차 | **측정된 영점** 상단 초과 |
| **E3 ★★★ 공동 주 관문(전역)** | `B_add − A_oracle` 전역 짝지은 차 | max(0, 측정된 영점 상단) 초과 |
| **E4** | 배포판 `B_add_em − A_em` (매크로·전역) | 참고 |
| **E5 ★★** | 셔플 대조 `B_add_oracle − B_add_shuf` | 이득이 **자기 burden 정렬**의 몫인가 |
| **E6** | 과적합 진단 — `B_int` vs `B_add` | 관문 아님 |
| **E7** | 결론 검산표 | R38 ⑦ · R39 ⑤ |

### 판정표

- **E2 ✅ · E3 ✅** → 방법 B 가 A 를 **두 지표 모두**에서 이긴다 → 남은 건 `π̂` 를 고치는 것
- **E2 ✅ · E3 미결** → B 는 **레코드 내 판별**을 개선한다(A 가 못 하는 일). 전역은 미결
- **E2 ❌** → B 의 이점이 없다 → 층② 처방은 A 계열

⚠️ **1차의 전역 수치는 단일 분할(TEST 20)의 것이라 LORO 수치와 직접 비교하지 않는다.**
⚠️ **새 데이터 0** — `svdb_data5.npz` 만.


In [ ]:
# CELL 0 — 공용 사전점검
import numpy as np

def decide(lo, hi, thr, direction):
    if direction not in (">", "<"):
        raise ValueError("direction 은 '>' 또는 '<'")
    if direction == ">":
        if lo > thr: return "✅ 지지"
        if hi < thr: return "❌ 기각"
    else:
        if hi < thr: return "✅ 지지"
        if lo > thr: return "❌ 기각"
    return "⚠️ 미결"

def mde(lo, hi):
    return (hi - lo) / 2.0 if np.isfinite(lo) and np.isfinite(hi) else float("nan")

def boot_mean(v, seed, nb=3000, q=2.5):
    d = np.asarray(v, float); d = d[np.isfinite(d)]
    if len(d) < 3:
        return float("nan"), float("nan"), float("nan"), len(d)
    rng = np.random.RandomState(seed)
    b = [d[rng.randint(0, len(d), len(d))].mean() for _ in range(nb)]
    return (float(d.mean()), float(np.percentile(b, q)),
            float(np.percentile(b, 100 - q)), len(d))

def boot_pair(a, b, seed, nb=3000, q=2.5):
    """★ **짝지은 차**(b − a) — 같은 레코드에서 두 팔을 재므로 짝을 유지한다."""
    a = np.asarray(a, float); b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b); a, b = a[m], b[m]
    if len(a) < 3:
        return float("nan"), float("nan"), float("nan"), len(a)
    rng = np.random.RandomState(seed)
    d = [(b[j] - a[j]).mean() for j in (rng.randint(0, len(a), len(a)) for _ in range(nb))]
    return (float((b - a).mean()), float(np.percentile(d, q)),
            float(np.percentile(d, 100 - q)), len(a))

def _rank_avg(v):
    v = np.asarray(v, float); o = v.argsort()
    r = np.empty(len(v), float); r[o] = np.arange(len(v), dtype=float)
    for u in np.unique(v):
        m = v == u
        if m.sum() > 1:
            r[m] = r[m].mean()
    return r

def spearman(a, b):
    """★★ 동점을 **평균 순위**로(Q3-B 에서 argsort 판본의 순서 의존이 드러났다)."""
    ra, rb = _rank_avg(a), _rank_avg(b)
    if np.std(ra) < 1e-12 or np.std(rb) < 1e-12:
        return float("nan")
    return float(np.corrcoef(ra, rb)[0, 1])

def need_super(n, half, eff, p80=False):
    if not np.isfinite(half) or abs(eff) < 1e-9 or n < 1:
        return float("nan")
    r = float(n) * (half / abs(eff)) ** 2
    return r * 2.04 if p80 else r

def ece_of(p, y, nbin=15):
    p = np.asarray(p, float); y = np.asarray(y, float)
    edges = np.linspace(0.0, 1.0, nbin + 1); e = 0.0
    for i in range(nbin):
        m = (p >= edges[i]) & (p < edges[i + 1] if i < nbin - 1 else p <= edges[i + 1])
        if m.any():
            e += m.mean() * abs(p[m].mean() - y[m].mean())
    return float(e)

def derangement(n, rng):
    for _ in range(1000):
        p = rng.permutation(n)
        if not np.any(p == np.arange(n)):
            return p
    return np.roll(np.arange(n), 1)

class AssetError(RuntimeError): pass
print("CELL 0 ✅")


In [ ]:
# CELL 1 — 설정 · 재사전등록
import os, sys, json, importlib, time, warnings
importlib.invalidate_caches(); warnings.filterwarnings("ignore")

SMOKE = os.environ.get("MEDKOS_SMOKE") == "1"
_ENV_ROOT = os.environ.get("MEDKOS_DRIVE_ROOT")
if _ENV_ROOT:
    DRIVE_ROOT = _ENV_ROOT
else:
    try:
        from google.colab import drive; drive.mount("/content/drive", force_remount=False)
        DRIVE_ROOT = "/content/drive/MyDrive"
    except Exception as e:
        print("⚠️ Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
MITBIH  = os.path.join(DRIVE_ROOT, "mitbih")
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun

SEED0, IDX_S = 20260804, 1
FS = 360
FULL_K = tuple(range(4, 33))
RHY_K  = (5, 10, 20, 32)
MIN_S, MIN_N = 25, 25

# ── ★ 사전등록 상수 (SMOKE 가 절대 안 건드린다)
TOL_IDENT = 1e-12
DEV_EVERY = 4            # LORO 안에서 나머지 레코드 중 4개마다 1개를 DEV 로(부담순)
NONINF_MARGIN = 0.01

# ── 비용 손잡이
NB_BOOT = 400 if SMOKE else 2000
N_SHUF  = 3   if SMOKE else 10
N_PERM  = 2   if SMOKE else 5      # ★ LORO 라 1 rep = 56 fold — reps 를 낮게 잡는다

# ★★★ 주 관문은 **`B_add` 대 A** 다(1차는 `B_int` 를 썼고 그게 틀렸다).
#     `B_int` 는 **과적합 진단**으로 강등한다(1차 실측 0.4116 ≪ B_add 0.5725).
ARMS = ("raw", "A_oracle", "A_em", "B_add_oracle", "B_add_em", "B_add_shuf", "B_int_oracle")
PRIMARY = ("macro", "pooled")      # ★ 공동 주 지표
READ_ORDER = ("E0", "E1", "E2", "E3", "E4", "E5", "E6", "E7")

SV5 = os.path.join(MITBIH, "svdb_data5.npz")

REF = dict(  # ⚠️ 1차는 **단일 분할(TEST 20)** — LORO 수치와 직접 비교하지 않는다
    q4_raw=0.3362, q4_A_or=0.5514, q4_B_add=0.5725, q4_B_int=0.4116,
    q4_macro_raw=0.5389, q4_macro_Badd=0.5787,
    q4_dmacro_Badd=0.039773, q4_dmacro_lo=0.005381, q4_dmacro_hi=0.074179,
    q4_d2b=0.0212, q4_rho_add=0.792062, q4_contrib_add=4.46e-16,
    dom_rec=48, dom_pi=0.5764)

RULE_CHECK = {
    "R11 매크로":       "★★★ 매크로를 **공동 주 지표로 승격** — A 가 원리적으로 못 움직이는 지표다",
    "R16 fallback 없음": "자산 없으면 **중단**",
    "R22 누출 없음":     "LORO 안에서 기저·보정을 **held-out 레코드를 빼고** 적합",
    "R26 / R38 ②":      "★★ **대비의 영점**을 rep×레코드로 측정한다. 못 쟀으면 문턱을 0 으로 "
                        "되돌리지 않고 **관문을 읽지 않는다**(스모크가 이 함정을 잡았다)",
    "R29 ② 분기 금지":   "E0 이 깨지면 아래를 **안 읽는다**",
    "R33 ① MDE":        "관문마다 MDE. **미결 ≠ 등가**",
    "R34 ③ 대조":       "★★ `B_add_shuf` 는 같은 기저·같은 값 집합, **대응만** 깨진다",
    "R35 ① 자 먼저":    "★★★ 1차의 이론(「상수 특징 = A 와 같은 구조」)이 **반증됐다** — E1 이 분리 보고",
    "R36 ② 선택 편의":  "★ 주 관문을 **사전등록 문구**(별도 특징)에 맞춰 `B_add` 로 고정",
    "R38 ⑦ 요약 정합":  "요약·검산표·판정이 같은 갈래여야 한다",
    "R39 ① 여유 고정":  f"매크로 비열등 여유 {NONINF_MARGIN} 를 사전 고정",
}

CONFIG = dict(
    exp="quest46_q4b_burden_feature_v2", quest="ailab-2026-0046", step="burden-feature-v2",
    parent_exp=["quest46_q4_burden_feature", "quest46_q3b_prior_shuffle"],
    purpose=("**주 관문 교체 + 검정력 확보.** 1차(`20260804T2349`)는 「상수 특징 = 방법 A 와 "
             "구조 동일」이라는 내 이론으로 `B_add` 를 주 관문에서 빼고 상호작용만 뒀는데, "
             "실측이 그 이론을 **반증**했다 — burden **기여분**이 레코드별 상수인 건 참"
             "(SD 4.46e-16)이지만, 특징을 추가하면 로지스틱이 **리듬 계수를 전부 다시 적합**"
             "하므로 레코드 내 판별이 바뀐다(raw 대비 ρ **0.792** · 매크로 **+0.0398**). "
             "결과적으로 **가장 잘한 팔(`B_add` 전역 0.5725)이 주 관문 밖에** 있었다. "
             "★★★ 그리고 1차의 진짜 발견은 **매크로**다 — B 계열 셋 다 raw 대비 CI 가 0 을 "
             "떼며 올랐고(비열등이 아니라 **우월**), 이건 **A 가 원리적으로 못 하는 일**이다"
             "(A 는 상수 시프트라 매크로가 정의상 불변 · 1차에서 A 매크로 = raw 매크로 = "
             "0.5389 로 완전 동일). 그래서 이 런은 주 관문을 `B_add` 대 A 로 바꾸고, "
             "**매크로를 공동 주 지표로 승격**하며, 1차에 없던 **`B_add_em`(배포판)** 을 넣고, "
             "**LORO** 로 56 레코드 전부를 평가에 쓴다(1차는 TEST 20 · 전역 MDE 0.1614)."),
    dataset="SVDB — svdb_data5.npz (리듬 특징만 · 새 데이터 0)",
    arms=list(ARMS), primary=list(PRIMARY), read_order=READ_ORDER,
    noninf_margin=NONINF_MARGIN, dev_every=DEV_EVERY,
    n_boot=NB_BOOT, n_shuf=N_SHUF, n_perm=N_PERM, smoke=SMOKE,
    ref=REF, rule_check=RULE_CHECK,
    predictions={
        "E0": "코호트 확인 + **항등** — A 팔은 레코드별 상수 시프트이므로 매크로가 raw 와 "
              "**정확히 같아야** 한다. 깨지면 구현이 틀린 것이므로 **중단**",
        "E1": "★★ **구조(관문 아님) — 두 명제를 분리해 보고한다.** ⓐ burden **기여분**의 "
              "레코드 내 산포(≈0 이면 상수) ⓑ 모델 **전체**의 raw 대비 레코드 내 ρ. "
              "1차는 ⓐ 에서 ⓑ 를 잘못 추론했다 — ⓑ 는 **재적합 효과**라 따로 재야 한다",
        "E2": "★★★ **공동 주 관문(매크로)** — `B_add_oracle − A_oracle` 매크로 짝지은 차. "
              "A 의 매크로는 raw 와 항등이므로 이건 곧 「B 가 **레코드 내 판별**을 "
              "개선하는가」다. **측정된 영점** 상단 초과가 기준",
        "E3": "★★★ **공동 주 관문(전역)** — `B_add_oracle − A_oracle` 전역 짝지은 차. "
              "사전등록 「Q3 대비 > 0」이므로 문턱은 **max(0, 측정된 영점 상단)**",
        "E4": "배포판 `B_add_em − A_em` (매크로·전역 둘 다). Q3 의 π̂ 한계를 물려받는다",
        "E5": "★★ **셔플 대조** `B_add_oracle − B_add_shuf` — 같은 기저·같은 burden 값 집합, "
              "**대응만** 깨진다. 이득이 자기 burden 정렬의 몫인지 가른다",
        "E6": "**과적합 진단(관문 아님)** — `B_int` 가 `B_add` 보다 나쁜가. 1차 실측 "
              "0.4116 ≪ 0.5725 였고, 상호작용 9개가 더 붙는다",
        "E7": "결론 검산표"},
    caveat=("★★★ **1차의 전역 수치는 단일 분할(TEST 20)의 것이라 LORO 수치와 직접 비교하지 "
            "않는다** — 재현 앵커가 아니라 참고값이다. "
            "★★ **주 관문을 바꾸는 이유를 명시한다**(R39 ①): 사후에 유리한 팔을 고른 게 "
            "아니라, 사전등록 문구가 「burden 을 **별도 특징**으로」인데 1차가 그걸 구조 대조로 "
            "밀어냈기 때문이다. `B_int` 는 그대로 두되 **과적합 진단**으로 강등한다. "
            "★ 오라클 팔은 **상한이지 방법이 아니다** — 배포판은 `*_em` 이다."))
np.random.seed(SEED0)
run = MedKOSRun("quest46_q4b_burden_feature_v2", CONFIG, project=PROJECT)
run.log("설정 ✅ **Q4-B — 주 관문을 고친다**")
run.log("  ★★★ 주 관문 = **`B_add` 대 A** (1차는 `B_int` 를 썼고 그게 틀렸다)")
run.log("  ★★★ **매크로를 공동 주 지표로 승격** — A 가 원리적으로 못 움직이는 지표다")
run.log(f"  ★★ **LORO** — 56 레코드 전부 평가(1차는 TEST 20 · 전역 MDE 0.1614)")
run.log("  ★ `B_add_em`(배포판) 추가 — 1차에 **없었다**")
if SMOKE:
    run.log(f"  ⚠️ **스모크런** — 비용 손잡이만 축소(NB_BOOT={NB_BOOT} · N_SHUF={N_SHUF} · "
            f"N_PERM={N_PERM}). 관문 문턱은 그대로다")
run.log("\n  재사전등록 규칙 체크리스트 (R29 ③)")
for k_, v_ in RULE_CHECK.items():
    run.log(f"    [x] {k_:<18} {v_}")


In [ ]:
# CELL 2 — 【E-0】 코호트 · LORO 골격 · 팔 정의
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import average_precision_score, roc_auc_score

run.log("\n" + "=" * 100)
run.log("【E-0】 코호트 · LORO 골격 · 팔 정의")
run.log("=" * 100)
VERD, DIFF = {}, {}
def g_(k, v, d):
    VERD[k] = v; run.log(f"  {k:<5}{v}  {d}")

if not os.path.exists(SV5):
    raise AssetError(f"{SV5} 없음(R16)")
D5 = np.load(SV5, allow_pickle=True)
PID = np.asarray(D5["pid"]).astype(int); Y3 = np.asarray(D5["y3"]).astype(int)
PRE = np.asarray(D5["pre_rr"], float); POST = np.asarray(D5["post_rr"], float)
K = np.where(Y3 >= 0)[0]
RID = PID[K]; Y = Y3[K]; TT = (Y == IDX_S)
pre = PRE[K].astype(float); post = POST[K].astype(float)
RS = np.array(sorted(set(RID.tolist())))

_S = pd.Series(pre); _G = _S.groupby(pd.Series(RID))
def local_base(k):
    r = np.asarray(_G.apply(lambda x: x.shift(1).rolling(k, min_periods=1).median())).astype(float)
    return np.where(np.isfinite(r), r, pre)
_med = _G.transform("median").to_numpy()
_std = _G.transform("std").to_numpy(); _mean = _G.transform("mean").to_numpy()
f1 = _med - pre
f2 = {k: 1.0 - pre / (local_base(k) + 1e-9) for k in FULL_K}
f3 = post - pre
f4 = np.nan_to_num(_std / (_mean + 1e-9))
RHY = np.nan_to_num(np.c_[f1, np.column_stack([f2[k] for k in RHY_K]), f3, f4,
                          np.log1p(np.clip(pre, 0, None)), np.log1p(np.clip(post, 0, None))],
                    nan=0.0, posinf=0.0, neginf=0.0)

IDXS = {int(r): np.where(RID == r)[0] for r in RS}
REC_OK = [int(r) for r in RS
          if TT[IDXS[int(r)]].sum() >= MIN_S and (~TT[IDXS[int(r)]]).sum() >= MIN_N]
BURD = {r: float(TT[IDXS[r]].mean()) for r in REC_OK}
s_all = np.array([int(TT[IDXS[r]].sum()) for r in REC_OK], float)
DOMINANT = float(s_all.max() / s_all.sum())
DOM_REC = int(REC_OK[int(np.argmax(s_all))])
run.log(f"  레코드 {len(RS)} · 채점 가능 **{len(REC_OK)}** · 제외 {len(RS)-len(REC_OK)}")
run.log(f"  유병률 {min(BURD.values()):.4f}~{max(BURD.values()):.4f} · "
        f"지배 지분 **{DOMINANT:.3f}**(레코드 {DOM_REC}) · **전역 단독 인용 금지**(R11)")
run.log("  ★★ **LORO** — 레코드 하나를 빼고 나머지로 학습·보정, 그 레코드에서 평가. "
        "1차는 TEST 20 레코드뿐이었다(전역 MDE 0.1614)")

# ── 보정기
def make_platt(s, y):
    lr = LogisticRegression(max_iter=3000, C=1e6)
    lr.fit(np.asarray(s).reshape(-1, 1), np.asarray(y).astype(int))
    return lambda v: lr.predict_proba(np.asarray(v).reshape(-1, 1))[:, 1]

def make_iso(s, y):
    ir = IsotonicRegression(out_of_bounds="clip", y_min=1e-6, y_max=1 - 1e-6)
    ir.fit(np.asarray(s), np.asarray(y).astype(float))
    return lambda v: np.clip(ir.predict(np.asarray(v)), 1e-6, 1 - 1e-6)

EPS = 1e-6
def logit(p):
    p = np.clip(np.asarray(p, float), 1e-12, 1 - 1e-12)
    return np.log(p) - np.log1p(-p)

def em_prior(p, pi_tr, iters=100, tol=1e-9, clip=1e-2):
    pi = float(pi_tr)
    for _ in range(int(iters)):
        w = pi / pi_tr; v = (1.0 - pi) / (1.0 - pi_tr)
        num = w * p
        pp = num / (num + v * (1.0 - p))
        new = float(np.clip(pp.mean(), clip, 1.0 - clip))
        if abs(new - pi) < tol:
            pi = new; break
        pi = new
    return pi

# ── ★ LORO 분할 — held-out 을 뺀 나머지를 부담순으로 세워 DEV_EVERY 마다 하나를 DEV 로
def split_rest(held):
    rest = sorted([r for r in REC_OK if r != held], key=lambda r: (BURD[r], r))
    dv = [r for i, r in enumerate(rest) if i % DEV_EVERY == 0]
    tr = [r for r in rest if r not in set(dv)]
    return tr, dv

def feats(idx, bvec, mode):
    if mode == "raw":
        return RHY[idx]
    if mode == "add":
        return np.c_[RHY[idx], bvec]
    return np.c_[RHY[idx], bvec, RHY[idx] * bvec[:, None]]

def bvec_of(idx, by_rec, default):
    return np.array([by_rec.get(int(r), default) for r in RID[idx]], float)

def loro_arm(arm, y_override=None, shuf_map=None, seed=0):
    """★ 레코드 하나를 빼고 학습·보정 → 그 레코드에서 **보정된 확률**을 낸다.
    모든 팔을 **보정 뒤** 비교하므로 fold 간 척도가 맞는다(전역 비교의 전제)."""
    out = np.full(len(K), np.nan)
    mode = "raw" if arm in ("raw", "A_oracle", "A_em") else \
           ("int" if arm.startswith("B_int") else "add")
    for held in REC_OK:
        tr_r, dv_r = split_rest(held)
        tr = np.concatenate([IDXS[r] for r in tr_r])
        dv = np.concatenate([IDXS[r] for r in dv_r])
        te = IDXS[held]
        ytr = (TT[tr].astype(int) if y_override is None else y_override[arm][held])
        # 학습·DEV 는 라벨이 있으므로 실측 burden 을 쓴다. held-out 만 팔마다 다르다
        # (oracle=실측 · em=무라벨 π̂ · shuf=대응 깨진 값).
        bt = dict(shuf_map) if shuf_map is not None else dict(BURD)
        mu = float(np.mean([BURD[r] for r in tr_r]))
        Ftr = feats(tr, bvec_of(tr, bt, mu), mode)
        fmu, fsd = Ftr.mean(0), Ftr.std(0) + 1e-9
        lr = LogisticRegression(max_iter=3000, C=1.0)
        lr.fit((Ftr - fmu) / fsd, ytr)
        sc = lambda ii, bv: lr.decision_function((feats(ii, bv, mode) - fmu) / fsd)
        s_dv = sc(dv, bvec_of(dv, bt, mu))
        cal = make_iso(s_dv, TT[dv])
        pi_tr = float(TT[dv].mean())
        if mode == "raw":
            p_te = np.clip(cal(sc(te, bvec_of(te, bt, mu))), EPS, 1 - EPS)
            l = logit(p_te)
            if arm == "A_oracle":
                l = l + (logit(BURD[held]) - logit(pi_tr))
            elif arm == "A_em":
                l = l + (logit(em_prior(p_te, pi_tr)) - logit(pi_tr))
            out[te] = l
        else:
            if arm.endswith("_em"):
                p0 = np.clip(cal(sc(te, np.full(len(te), mu))), EPS, 1 - EPS)
                bh = em_prior(p0, pi_tr)          # ★ 무라벨 추정으로 held-out burden 을 채운다
            elif shuf_map is not None:
                bh = shuf_map[held]
            else:
                bh = BURD[held]
            out[te] = logit(np.clip(cal(sc(te, np.full(len(te), bh))), EPS, 1 - EPS))
    return out

pooled_of = lambda L: float(average_precision_score(TT[np.isfinite(L)].astype(int),
                                                    L[np.isfinite(L)]))
def per_rec(L):
    d = {}
    for r in REC_OK:
        pos = IDXS[r]; yy = TT[pos].astype(int)
        if 0 < yy.sum() < len(yy) and np.all(np.isfinite(L[pos])):
            d[r] = float(average_precision_score(yy, L[pos]))
    return d
macro_of = lambda L: float(np.mean(list(per_rec(L).values())))
run.log("  팔 정의 완료 — 모든 팔이 **보정 뒤** 로짓으로 비교된다")
CONFIG["cohort"] = dict(n_rec=len(RS), n_ok=len(REC_OK), dominant=DOMINANT,
                        dom_rec=DOM_REC, dev_every=DEV_EVERY)
run.save_json("config", CONFIG)


In [ ]:
# CELL 3 — 【E-A】 팔 실행 · E0 항등
run.log("\n" + "=" * 100)
run.log("【E-A】 LORO 실행 · E0 — A 팔의 매크로 항등")
run.log("=" * 100)
T0 = time.time()
perm0 = derangement(len(REC_OK), np.random.RandomState(SEED0 + 300))
SHUF0 = {REC_OK[i]: BURD[REC_OK[perm0[i]]] for i in range(len(REC_OK))}

L = {}
for a in ARMS:
    L[a] = loro_arm(a, shuf_map=(SHUF0 if a == "B_add_shuf" else None))
    run.log(f"  ({time.time()-T0:>5.0f}초) {a} 완료")
POOL = {a: pooled_of(L[a]) for a in ARMS}
PER = {a: per_rec(L[a]) for a in ARMS}
MACRO = {a: float(np.mean(list(PER[a].values()))) for a in ARMS}
run.log(f"\n  {'팔':<15}{'전역 PR-AUC':>13}{'매크로':>10}{'n':>5}")
for a in ARMS:
    run.log(f"  {a:<15}{POOL[a]:>13.4f}{MACRO[a]:>10.4f}{len(PER[a]):>5}")
run.log(f"  (참고 1차 단일분할 — raw {REF['q4_raw']} · A_or {REF['q4_A_or']} · "
        f"B_add {REF['q4_B_add']} · B_int {REF['q4_B_int']})")
run.log("  ⚠️ 1차는 **TEST 20 레코드**의 수라 LORO 와 직접 비교하지 않는다")

# ── E0 항등 — A 팔은 레코드별 상수 시프트라 매크로가 raw 와 **정확히** 같아야 한다
d_ident = max(abs(MACRO["A_oracle"] - MACRO["raw"]), abs(MACRO["A_em"] - MACRO["raw"]))
run.log(f"\n  E0 — A 팔 매크로 vs raw 의 max|Δ| = **{d_ident:.2e}** (허용 {TOL_IDENT:.0e})")
if d_ident >= TOL_IDENT:
    raise AssetError(f"E0 실패({d_ident:.3e}) — A 는 레코드별 상수 시프트라 매크로가 불변이어야 "
                     "한다. 깨졌다면 구현이 틀린 것이므로 아래를 읽지 않는다(R29 ②)")
g_("E0", "✅ 지지",
   f"A 팔의 매크로가 raw 와 **정확히 같다**({d_ident:.1e}) — 상수 시프트가 구성으로 확인됐다")
CONFIG["E0"] = dict(pooled=POOL, macro=MACRO, ident=float(d_ident),
                    n_per={a: len(PER[a]) for a in ARMS})
run.save_json("config", CONFIG)


In [ ]:
# CELL 4 — 【E-B】 ★★ E1 — 구조: **두 명제를 분리**해 보고한다
run.log("\n" + "=" * 100)
run.log("【E-B】 E1 — 구조(관문 아님): 기여분은 상수인가 · 모델 전체는 어떤가")
run.log("=" * 100)
run.log("  ★★★ 1차는 여기서 **참인 명제에서 거짓인 결론**을 끌어냈다:")
run.log("     ⓐ 「burden **기여분**이 레코드별 상수다」 → **참**(1차 실측 SD 4.46e-16)")
run.log("     ⓑ 「따라서 모델 전체가 A 와 같은 구조다」 → **거짓**(1차 실측 ρ 0.792 · 매크로 +0.0398)")
run.log("     ▸ 특징을 추가하면 로지스틱이 **리듬 계수를 전부 다시 적합**한다. burden 항이")
run.log("       상수여도 나머지 가중치가 재조정돼 **레코드 내 판별이 바뀐다**")

# ⓐ 기여분이 상수인가 — burden 을 상수로 바꿨을 때의 점수 차, 레코드 내 산포
run.log("\n  ⓐ burden **기여분**의 레코드 내 산포 (0 이면 레코드별 상수)")
CONTRIB = {}
for a, mode in (("B_add_oracle", "add"), ("B_int_oracle", "int")):
    within, tot = [], []
    for held in REC_OK[:12]:                      # 진단이므로 앞 12 레코드로 충분
        tr_r, dv_r = split_rest(held)
        tr = np.concatenate([IDXS[r] for r in tr_r]); te = IDXS[held]
        mu = float(np.mean([BURD[r] for r in tr_r]))
        Ftr = feats(tr, bvec_of(tr, BURD, mu), mode)
        fmu, fsd = Ftr.mean(0), Ftr.std(0) + 1e-9
        lr = LogisticRegression(max_iter=3000, C=1.0).fit((Ftr - fmu) / fsd, TT[tr].astype(int))
        f_ = lambda bv: lr.decision_function((feats(te, bv, mode) - fmu) / fsd)
        d = f_(np.full(len(te), BURD[held])) - f_(np.full(len(te), mu))
        within.append(float(d.std())); tot.append(float(abs(d.mean())))
    CONTRIB[a] = dict(within=float(np.mean(within)), scale=float(np.mean(tot)))
    run.log(f"    {a:<15} 레코드 내 SD **{np.mean(within):.3e}** · 시프트 크기 "
            f"{np.mean(tot):.3e}  "
            + ("(상수 = A 와 **같은 항**)" if np.mean(within) < 1e-9 else "← 레코드 안에서 변한다"))

# ⓑ 모델 **전체**는 raw 의 레코드 내 순위를 보존하는가 — 1차가 오해한 자리
run.log("\n  ⓑ 모델 **전체**의 raw 대비 레코드 내 ρ (1.0 이면 순위 보존)")
RHO_IN, RHO_N = {}, {}
for a in ("A_oracle", "B_add_oracle", "B_int_oracle"):
    vv = np.array([spearman(L["raw"][IDXS[r]], L[a][IDXS[r]]) for r in REC_OK], float)
    ok = np.isfinite(vv)
    # ★ 스모크(널 조건)가 잡았다: 신호가 없으면 등장성 보정이 **상수로 붕괴**해 레코드 내
    #   점수가 전부 같아지고 ρ 가 정의되지 않는다. nan 을 평균에 흘리면 「✅ 분리됐다」가
    #   근거 없이 찍힌다 — 유효 레코드 수를 함께 보고하고 판정에 반영한다.
    RHO_IN[a] = float(np.mean(vv[ok])) if ok.any() else float("nan")
    RHO_N[a] = int(ok.sum())
    tag = "(A 는 상수 시프트라 **정확히 1.0**)" if a == "A_oracle" else \
          ("← **순위가 크게 바뀐다** = 재적합 효과"
           if np.isfinite(RHO_IN[a]) and RHO_IN[a] < 0.999 else "")
    run.log(f"    {a:<15} ρ **{RHO_IN[a]:.6f}**  (유효 {RHO_N[a]}/{len(REC_OK)}){tag}")
run.log(f"    (참고 1차 `B_add` ρ {REF['q4_rho_add']})")
if RHO_N["B_add_oracle"] < len(REC_OK):
    run.log(f"    ⚠️ {len(REC_OK)-RHO_N['B_add_oracle']}개 레코드에서 ρ 가 **정의되지 않았다** "
            "— 보정이 상수로 붕괴해 레코드 내 점수가 전부 같다")
run.log("\n  ▸ ⓐ 와 ⓑ 는 **다른 명제**다 — 기여분이 상수여도 모델 전체는 달라진다.")
run.log("    1차의 검산표는 이 둘을 구분하지 않아 「B_add = A」로 단언했다(R38 ⑦)")
_r = RHO_IN["B_add_oracle"]; _c = CONTRIB["B_add_oracle"]["within"]
if not np.isfinite(_r) or RHO_N["B_add_oracle"] < max(3, len(REC_OK) // 2):
    g_("E1", "⚠️ 미결",
       f"ρ 를 잰 레코드가 {RHO_N['B_add_oracle']}/{len(REC_OK)} 뿐이다 — 두 명제의 분리를 "
       "**이 런에서는 확인하지 못했다**(기여분 SD 는 여전히 "
       f"{_c:.1e})")
elif _r < 0.999:
    g_("E1", "✅ 지지",
       f"기여분은 상수(`B_add` SD {_c:.1e})인데 모델 전체는 순위를 바꾼다"
       f"(ρ {_r:.4f} · 유효 {RHO_N['B_add_oracle']}) — **두 명제가 분리됐다**")
else:
    g_("E1", "❌ 기각",
       f"모델 전체도 raw 의 레코드 내 순위를 보존한다(ρ {_r:.6f}) — 이 데이터에선 "
       f"`B_add` 가 정말 A 와 같은 구조다. 1차(ρ {REF['q4_rho_add']})와 갈린다")
CONFIG["E1"] = dict(contrib=CONTRIB, rho_within=RHO_IN, rho_n=RHO_N)
run.save_json("config", CONFIG)


In [ ]:
# CELL 5 — 【E-C】 ★★★ E2 매크로 · E3 전역 (공동 주 관문) · E4 · E5 · E6
run.log("\n" + "=" * 100)
run.log("【E-C】 E2(매크로) · E3(전역) — **공동 주 관문** · E4 배포판 · E5 셔플 · E6 과적합")
run.log("=" * 100)

def boot_pooled(arms, seed, nb):
    """레코드 군집 부트스트랩 — 모든 팔을 **같은 재표집**에 태운다."""
    rng = np.random.RandomState(seed)
    out = {k: [] for k in arms}
    for _ in range(nb):
        pick = [REC_OK[i] for i in rng.randint(0, len(REC_OK), len(REC_OK))]
        pos = np.concatenate([IDXS[r] for r in pick])
        yy = TT[pos].astype(int)
        if yy.sum() < 5 or yy.sum() == len(yy):
            continue
        for k, LL in arms.items():
            out[k].append(float(average_precision_score(yy, LL[pos])))
    return {k: np.asarray(v, float) for k, v in out.items()}

T1 = time.time()
BP = boot_pooled({a: L[a] for a in ARMS}, SEED0 + 11, NB_BOOT)
run.log(f"  ({time.time()-T1:.0f}초) 전역 부트스트랩 {len(BP['raw'])}회")

def pooled_diff(a, b):
    d = BP[b] - BP[a]
    return dict(mean=POOL[b] - POOL[a], lo=float(np.percentile(d, 2.5)),
                hi=float(np.percentile(d, 97.5)),
                mde=float(mde(float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5)))))

def macro_diff(a, b, seed):
    ks = [r for r in REC_OK if r in PER[a] and r in PER[b]]
    m_, lo_, hi_, n_ = boot_pair([PER[a][r] for r in ks], [PER[b][r] for r in ks], seed, NB_BOOT)
    return dict(mean=m_, lo=lo_, hi=hi_, n=int(n_), mde=float(mde(lo_, hi_)))

E2 = macro_diff("A_oracle", "B_add_oracle", SEED0 + 41)
E3 = pooled_diff("A_oracle", "B_add_oracle")
E4m = macro_diff("A_em", "B_add_em", SEED0 + 42)
E4p = pooled_diff("A_em", "B_add_em")
E5m = macro_diff("B_add_shuf", "B_add_oracle", SEED0 + 43)
E5p = pooled_diff("B_add_shuf", "B_add_oracle")
E6m = macro_diff("B_int_oracle", "B_add_oracle", SEED0 + 44)
E6p = pooled_diff("B_int_oracle", "B_add_oracle")
run.log(f"\n  {'대비':<30}{'매크로 Δ':>26}{'전역 Δ':>26}")
for nm, dm, dp in (("E2  B_add − A (oracle)", E2, E3), ("E4  B_add − A (em)", E4m, E4p),
                   ("E5  B_add − B_add_shuf", E5m, E5p), ("E6  B_add − B_int", E6m, E6p)):
    run.log(f"  {nm:<30}{dm['mean']:>+9.4f} [{dm['lo']:+.4f},{dm['hi']:+.4f}]"
            f"{dp['mean']:>+9.4f} [{dp['lo']:+.4f},{dp['hi']:+.4f}]")
run.log(f"    ▸ 성분 — 매크로 raw {MACRO['raw']:.4f} · A {MACRO['A_oracle']:.4f} · "
        f"B_add {MACRO['B_add_oracle']:.4f} | 전역 raw {POOL['raw']:.4f} · "
        f"A {POOL['A_oracle']:.4f} · B_add {POOL['B_add_oracle']:.4f} (R36 ⑤)")

# ── ★★ 대비의 영점 (라벨 치환 · 0 을 가정하지 않는다)
#    ⚠️ 스모크가 잡은 함정: rep 당 **한 숫자**만 모으면 n=N_PERM 이라 CI 가 안 나오고
#       (boot 은 n<3 에서 nan) 문턱이 **조용히 0 으로 되돌아간다** — 그건 「영점을
#       측정한다」가 아니라 「0 을 가정한다」다(R26·R38 ②). 그래서 **rep×레코드**로
#       보관해 레코드 군집 부트스트랩을 태운다(G1′ 에서 쓴 것과 같은 수선).
run.log(f"\n  ★★ **대비의 영점** — 학습 라벨 치환 뒤 같은 대비 (reps={N_PERM})")
NUL_PER = {}                    # rec -> [rep 별 (B_add − A) 레코드 내 PR-AUC 차]
NUL_L = []                      # rep -> (la, lb) 로짓 (전역 영점의 군집 부트스트랩용)
for s_ in range(N_PERM):
    rr = np.random.RandomState(SEED0 + 400 + s_)
    yov = {}
    for a in ("A_oracle", "B_add_oracle"):
        yov[a] = {}
        for held in REC_OK:
            tr_r, _ = split_rest(held)
            tr = np.concatenate([IDXS[r] for r in tr_r])
            yov[a][held] = TT[tr].astype(int)[rr.permutation(len(tr))]
    la = loro_arm("A_oracle", y_override=yov)
    lb = loro_arm("B_add_oracle", y_override=yov)
    pa, pb = per_rec(la), per_rec(lb)
    for r in REC_OK:
        if r in pa and r in pb:
            NUL_PER.setdefault(r, []).append(pb[r] - pa[r])
    NUL_L.append((la, lb))
    run.log(f"    ({time.time()-T1:>5.0f}초) 영점 rep {s_+1}/{N_PERM} — "
            f"매크로 {np.mean([pb[r]-pa[r] for r in pa if r in pb]):+.4f} · "
            f"전역 {pooled_of(lb)-pooled_of(la):+.4f}")

# 매크로 영점 — rep 평균을 레코드 단위 값으로 삼고 **레코드**를 재표집한다
nul_rec = {r: float(np.mean(v)) for r, v in NUL_PER.items() if len(v)}
NM = boot_mean(list(nul_rec.values()), SEED0 + 61, NB_BOOT)
# 전역 영점 — 같은 재표집(레코드 군집)을 모든 rep 에 태우고 rep 평균을 낸다
_rngN = np.random.RandomState(SEED0 + 62); _nb = []
for _ in range(NB_BOOT):
    pick = [REC_OK[i] for i in _rngN.randint(0, len(REC_OK), len(REC_OK))]
    pos = np.concatenate([IDXS[r] for r in pick]); yy = TT[pos].astype(int)
    if yy.sum() < 5 or yy.sum() == len(yy):
        continue
    _nb.append(float(np.mean([average_precision_score(yy, lb[pos])
                              - average_precision_score(yy, la[pos])
                              for la, lb in NUL_L])))
_nb = np.asarray(_nb, float)
NP = (float(np.mean([pooled_of(lb) - pooled_of(la) for la, lb in NUL_L])),
      float(np.percentile(_nb, 2.5)), float(np.percentile(_nb, 97.5)), len(_nb)) \
     if len(_nb) >= 3 else (float("nan"),) * 3 + (len(_nb),)
run.log(f"    매크로 영점 **{NM[0]:+.4f}** [{NM[1]:+.4f}, {NM[2]:+.4f}] "
        f"(레코드 {NM[3]} × rep {N_PERM})")
run.log(f"    전역   영점 **{NP[0]:+.4f}** [{NP[1]:+.4f}, {NP[2]:+.4f}] "
        f"(군집 부트 {NP[3]} × rep {N_PERM})")

# ★ 영점을 못 쟀으면 **문턱을 0 으로 되돌리지 않는다** — 관문을 읽지 않는다
NUL_OK = bool(np.isfinite(NM[2]) and np.isfinite(NP[2]))
E2_THR = NM[2] if np.isfinite(NM[2]) else float("nan")
E3_THR = max(0.0, NP[2]) if np.isfinite(NP[2]) else float("nan")
run.log(f"    ▸ E2 문턱 = 매크로 영점 상단 **{E2_THR:+.4f}**")
run.log(f"    ▸ E3 문턱 = max(0, 전역 영점 상단) **{E3_THR:+.4f}** (사전등록 「> 0」 + 영점)")
if not NUL_OK:
    run.log("    ⛔ **영점을 측정하지 못했다** — 0 으로 되돌리는 건 「측정」이 아니라 "
            "「가정」이다(R26). E2·E3 을 **읽지 않는다**")

e2_v = decide(E2["lo"], E2["hi"], E2_THR, ">") if NUL_OK else "⚠️ 미결"
e3_v = decide(E3["lo"], E3["hi"], E3_THR, ">") if NUL_OK else "⚠️ 미결"
_why = "" if NUL_OK else " — ★ **영점 미측정**이라 판정 불가(R26)"
g_("E2", e2_v,
   (f"매크로 — B 가 **레코드 내 판별**을 {E2['mean']:+.4f} 개선한다"
    f"(A 는 원리적으로 못 하는 일)" if e2_v.startswith("✅") else
    f"매크로 Δ {E2['mean']:+.4f} [{E2['lo']:+.4f}, {E2['hi']:+.4f}] · "
    f"MDE {E2['mde']:.4f} · 문턱 {E2_THR:+.4f} — "
    + ("**B 가 A 에 못 미친다**" if e2_v.startswith("❌") else "**등가가 아니다**")) + _why)
g_("E3", e3_v,
   (f"전역 — B 가 A 를 {E3['mean']:+.4f} 앞선다" if e3_v.startswith("✅") else
    f"전역 Δ {E3['mean']:+.4f} [{E3['lo']:+.4f}, {E3['hi']:+.4f}] · "
    f"MDE {E3['mde']:.4f} · 문턱 {E3_THR:+.4f} — "
    + ("**B 가 A 에 못 미친다**" if e3_v.startswith("❌") else "**등가가 아니다**")) + _why)

# ★ 셔플 대조의 영점은 **구성으로** 0 이다(같은 기저·같은 값 집합, 대응만 깨진다).
#   ★★ 주 지표가 둘이므로 E5 도 **둘 다** 읽는다(R40 ② — 판정과 비교에 같은 통계).
e5m_v = decide(E5m["lo"], E5m["hi"], 0.0, ">")
e5p_v = decide(E5p["lo"], E5p["hi"], 0.0, ">")
e5_v = "✅ 지지" if (e5m_v.startswith("✅") or e5p_v.startswith("✅")) else \
       ("❌ 기각" if (e5m_v.startswith("❌") and e5p_v.startswith("❌")) else "⚠️ 미결")
_e5_ax = ("매크로" if e5m_v.startswith("✅") else "") + \
         ("·" if e5m_v.startswith("✅") and e5p_v.startswith("✅") else "") + \
         ("전역" if e5p_v.startswith("✅") else "")
g_("E5", e5_v,
   f"이득이 **자기 burden 정렬**의 몫이다 — {_e5_ax}에서 "
   f"(매크로 {E5m['mean']:+.4f} {e5m_v} · 전역 {E5p['mean']:+.4f} {e5p_v})"
   if e5_v.startswith("✅") else
   (f"★★ 대응을 깨는 게 **오히려 낫다** — burden 정렬이 해롭다 "
    f"(매크로 {E5m['mean']:+.4f} · 전역 {E5p['mean']:+.4f})" if e5_v.startswith("❌") else
    f"매크로 {E5m['mean']:+.4f} [{E5m['lo']:+.4f}, {E5m['hi']:+.4f}] {e5m_v} · "
    f"전역 {E5p['mean']:+.4f} [{E5p['lo']:+.4f}, {E5p['hi']:+.4f}] {e5p_v} — "
    "정렬의 몫인지 **가르지 못했다**(미결 ≠ 등가 · R33 ①)"))

# ★ E6 은 **CI 로** 읽는다 — 점추정 부호로 판정하면 G1 의 G5 오류를 되풀이한다
e6_v = decide(E6m["lo"], E6m["hi"], 0.0, ">")
g_("E6", e6_v,
   f"`B_add` 가 `B_int` 를 매크로 {E6m['mean']:+.4f} · 전역 {E6p['mean']:+.4f} 앞선다 — "
   "상호작용은 **과적합**이다(1차 결론과 같다)" if e6_v.startswith("✅") else
   (f"`B_int` 가 `B_add` 를 앞선다(매크로 {E6m['mean']:+.4f}) — 1차의 과적합 해석을 뒤집는다"
    if e6_v.startswith("❌") else
    f"매크로 {E6m['mean']:+.4f} [{E6m['lo']:+.4f}, {E6m['hi']:+.4f}] · "
    f"전역 {E6p['mean']:+.4f} — **가르지 못했다**. 1차의 큰 격차"
    f"({REF['q4_B_add']}-{REF['q4_B_int']})가 재현되지 않았다"))
CONFIG["E2"] = E2; CONFIG["E3"] = E3; CONFIG["E4"] = dict(macro=E4m, pooled=E4p)
CONFIG["E5"] = dict(macro=E5m, pooled=E5p, v_macro=e5m_v, v_pooled=e5p_v)
CONFIG["E6"] = dict(macro=E6m, pooled=E6p)
CONFIG["null_diff"] = dict(macro=dict(mean=NM[0], lo=NM[1], hi=NM[2], n=int(NM[3])),
                           pooled=dict(mean=NP[0], lo=NP[1], hi=NP[2], n=int(NP[3])),
                           n_perm=N_PERM, measured=bool(NUL_OK),
                           e2_thr=float(E2_THR), e3_thr=float(E3_THR))
run.save_json("config", CONFIG)


In [ ]:
# CELL 6 — 【E-D】 필요표본 · 지배 레코드 진단 · ★ E7 검산표
run.log("\n" + "=" * 100)
run.log("【E-D】 필요표본 · 진단 · E7 결론 검산표")
run.log("=" * 100)
run.log(f"  필요표본 (**우월 프레임** · 단위 = 레코드 · 현재 {len(REC_OK)}개 · 측정된 영점 기준)")
NEED = {}
run.log(f"  {'대비':<12}{'효과-영점':>10}{'반폭':>9}{'n(50%)':>9}{'n(80%)':>9}")
for nm, d, thr in (("E2 매크로", E2, NM[0]), ("E3 전역", E3, NP[0])):
    eff = d["mean"] - thr
    n5 = need_super(len(REC_OK), d["mde"], eff, False)
    n8 = need_super(len(REC_OK), d["mde"], eff, True)
    # ★ NaN 을 「읽을 수 있다」로 흘리지 않는다 — 영점을 못 쟀으면 필요표본도 없다
    if not np.isfinite(eff):
        tag, zero = "★ **영점 미측정** — 필요표본을 계산할 수 없다(R26)", True
    elif abs(eff) < d["mde"]:
        tag, zero = "★ **효과 ≈ 0 이라 해석 불가**(R41 ②)", True
    else:
        tag, zero = "읽을 수 있다", False
    NEED[nm] = dict(effect=float(eff), half=float(d["mde"]), sup50=float(n5), sup80=float(n8),
                    uninterpretable=bool(zero))
    run.log(f"  {nm:<12}{eff:>+10.4f}{d['mde']:>9.4f}{n5:>9.0f}{n8:>9.0f}  {tag}")
run.log(f"    ▸ 1차는 TEST 20 레코드로 전역 MDE 0.1614 였다 → 이번 {E3['mde']:.4f}")

run.log(f"\n  ★ 진단 — 지배 레코드 {DOM_REC}(유병률 {BURD[DOM_REC]:.4f})의 레코드 내 PR-AUC")
for a in ARMS:
    v = PER[a].get(DOM_REC, float("nan"))
    tag = "  ← A 는 상수 시프트라 raw 와 같아야 한다" if a.startswith("A_") else ""
    run.log(f"  {a:<15}{v:>10.4f}{tag}")

run.log("\n  ★ E7 — **결론 검산표**")
CHECK = [
    dict(claim=f"E0 항등 — A 팔 매크로가 raw 와 정확히 같다({CONFIG['E0']['ident']:.1e})",
         num="A 는 레코드별 상수 시프트다 — 매크로는 정의상 불변",
         assume="**없음** — 구성으로 보장되고 런타임에 검사한다",
         iffalse="—  ★ 이것이 **A 가 매크로를 못 움직인다**는 것의 증명이고, E2 의 의미다"),
    dict(claim=f"E1 — 「기여분은 상수」와 「모델 전체가 A 와 같다」는 **다른 명제**다 "
               f"→ {VERD['E1']}",
         num=f"기여분 레코드 내 SD {CONFIG['E1']['contrib']['B_add_oracle']['within']:.1e} "
             f"vs 모델 전체 ρ {CONFIG['E1']['rho_within']['B_add_oracle']:.4f} "
             f"(유효 {CONFIG['E1']['rho_n']['B_add_oracle']}/{len(REC_OK)})",
         assume="**없음** — 둘을 따로 쟀다",
         iffalse=f"★ 1차는 앞에서 뒤를 추론해 `B_add` 를 주 관문에서 뺐고, 결과적으로 "
                 f"**가장 잘한 팔이 주 관문 밖에** 있었다(1차 ρ {REF['q4_rho_add']})"),
    dict(claim=f"E2 매크로 {E2['mean']:+.4f} [{E2['lo']:+.4f}, {E2['hi']:+.4f}] → {VERD['E2']}",
         num=f"영점 {NM[0]:+.4f} [{NM[1]:+.4f}, {NM[2]:+.4f}] (레코드 {NM[3]} × rep "
             f"{N_PERM}) · MDE {E2['mde']:.4f} · 레코드 {E2['n']}",
         assume="오라클 burden 이 **레코드 안에서 상수**라는 것",
         iffalse="유병률이 기록 안에서 표류하면 오라클조차 상한이 아니다"),
    dict(claim=f"영점을 **측정**했다 — {'예' if NUL_OK else '**아니오**'}",
         num=f"rep {N_PERM} × 레코드 {len(REC_OK)} 로 보관해 군집 부트스트랩을 태웠다"
             if NUL_OK else "CI 가 나오지 않았다",
         assume="**없음** — 못 쟀으면 문턱을 0 으로 되돌리지 않고 관문을 **읽지 않는다**",
         iffalse="★ 0 으로 되돌리면 「측정」이 아니라 「가정」이다(R26 · R38 ②). "
                 "rep 당 한 숫자만 모으면 n=N_PERM 이라 CI 가 안 나와 조용히 그렇게 된다 "
                 "— 스모크가 잡은 함정이다"),
    dict(claim=f"E3 전역 {E3['mean']:+.4f} [{E3['lo']:+.4f}, {E3['hi']:+.4f}]",
         num=f"문턱 max(0, 영점 상단) = {E3_THR:+.4f} · MDE {E3['mde']:.4f}",
         assume="LORO fold 간 점수가 **보정으로 같은 척도**에 있다는 것",
         iffalse="보정이 fold 마다 어긋나면 전역 비교가 흔들린다 — 그래서 **모든 팔을 "
                 "보정 뒤에** 비교한다"),
    dict(claim=f"E5 셔플 대조 매크로 {E5m['mean']:+.4f} · 전역 {E5p['mean']:+.4f}",
         num=f"같은 기저·같은 burden 값 집합, **대응만** 깨진다 · derangement",
         assume="유병률이 셔플로 재배치 가능할 만큼 다양하다는 것",
         iffalse=f"실측 범위 {min(BURD.values()):.4f}~{max(BURD.values()):.4f} 로 벌어져 있다"),
    dict(claim="주 관문을 1차에서 바꿨다",
         num="사전등록 문구가 「burden 을 **별도 특징**으로」인데 1차가 그걸 구조 대조로 밀어냈다",
         assume="**없음** — 사후에 유리한 팔을 고른 게 아니라 **사전등록 문구로 되돌린 것**이다",
         iffalse="⚠️ 그래도 1차 결과를 보고 바꾼 것이므로, **이 런의 수치로만** 판정한다"),
    dict(claim=f"전역이 보조 지표다 (지배 지분 {DOMINANT:.3f})",
         num=f"레코드 {len(REC_OK)} · LORO · **전역 단독 인용 금지**(R11)",
         assume="Q3-B 가 셔플 대조로 전역 지표를 검증했다는 것",
         iffalse="매크로가 R11 이 원래 요구한 지표이고, 이번엔 **공동 주 지표**로 올렸다"),
]
for i, c in enumerate(CHECK, 1):
    run.log(f"\n  [{i}] **{c['claim']}**")
    run.log(f"      근거   {c['num']}")
    run.log(f"      가정   {c['assume']}")
    run.log(f"      틀리면 {c['iffalse']}")
CONFIG["need"] = NEED; CONFIG["E7"] = CHECK
run.save_json("config", CONFIG)


In [ ]:
# CELL 7 — 【E-E】 그림 · 요약 · 마무리
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display
fig, ax = plt.subplots(1, 3, figsize=(16.5, 4.6))
EN = {"raw": "raw", "A_oracle": "A oracle", "A_em": "A em", "B_add_oracle": "B add or",
      "B_add_em": "B add em", "B_add_shuf": "B add shuf", "B_int_oracle": "B int or"}
xs = np.arange(len(ARMS))
ax[0].bar(xs - 0.2, [POOL[a] for a in ARMS], width=0.4, color="tab:blue", label="pooled")
ax[0].bar(xs + 0.2, [MACRO[a] for a in ARMS], width=0.4, color="tab:green", label="macro")
ax[0].axhline(MACRO["raw"], ls="--", color="k", lw=1.0)
ax[0].set_xticks(xs); ax[0].set_xticklabels([EN[a] for a in ARMS], fontsize=7, rotation=20)
ax[0].set_ylabel("PR-AUC")
ax[0].set_title(f"arms (LORO, n={len(REC_OK)})", fontsize=9)
ax[0].legend(fontsize=7); ax[0].grid(alpha=.3, axis="y")

nm = ["E2 macro", "E3 pooled", "E4m em", "E5m shuf", "E6m vs int"]
vv = [E2["mean"], E3["mean"], E4m["mean"], E5m["mean"], E6m["mean"]]
lo = [vv[0] - E2["lo"], vv[1] - E3["lo"], vv[2] - E4m["lo"], vv[3] - E5m["lo"], vv[4] - E6m["lo"]]
hi = [E2["hi"] - vv[0], E3["hi"] - vv[1], E4m["hi"] - vv[2], E5m["hi"] - vv[3], E6m["hi"] - vv[4]]
ax[1].errorbar(vv, np.arange(5), xerr=[lo, hi], fmt="o", capsize=5, color="tab:red")
ax[1].axvline(0, color="k", lw=.9)
if np.isfinite(E2_THR):
    ax[1].axvline(E2_THR, ls=":", color="tab:gray", lw=1.2, label=f"E2 thr {E2_THR:+.3f}")
else:
    ax[1].plot([], [], " ", label="E2 thr: null NOT measured")
ax[1].set_yticks(range(5)); ax[1].set_yticklabels(nm, fontsize=8)
ax[1].set_xlabel("paired difference (B_add - comparator)")
ax[1].set_title("contrasts", fontsize=9)
ax[1].legend(fontsize=7); ax[1].grid(alpha=.3, axis="x")

ks = sorted(PER["A_oracle"].keys())
ax[2].scatter([PER["A_oracle"][r] for r in ks], [PER["B_add_oracle"][r] for r in ks],
              s=28, color="tab:purple")
lim = [0.0, 1.0]
ax[2].plot(lim, lim, "k--", lw=.9)
ax[2].set_xlabel("A oracle (per record)"); ax[2].set_ylabel("B add oracle (per record)")
ax[2].set_title("E2 : per-record, macro is where A cannot move", fontsize=9)
ax[2].grid(alpha=.3)
fig.tight_layout()
PNG = run.save_fig("q4b_burden_feature_v2", fig)
plt.close(fig); display(Image(PNG))

run.log("\n" + "=" * 100)
run.log("요약")
run.log("=" * 100)
ok_ = lambda k: VERD.get(k, "").startswith("✅")
no_ = lambda k: VERD.get(k, "").startswith("❌")
for g in READ_ORDER[:6]:
    run.log(f"  {g:<5}{VERD.get(g, '(관문 아님)')}")
run.log("")
run.log(f"  LORO {len(REC_OK)}레코드 — " + " · ".join(f"{a} {POOL[a]:.4f}/{MACRO[a]:.4f}"
                                                     for a in ARMS))
run.log(f"  (전역/매크로) · 매크로 영점 {NM[0]:+.4f} [{NM[1]:+.4f}, {NM[2]:+.4f}] · "
        f"전역 영점 {NP[0]:+.4f} [{NP[1]:+.4f}, {NP[2]:+.4f}]")
run.log("")
# ★ 세 갈래 — **미결을 기각으로 쓰지 않는다**(R29 ① · R33 ①). 요약·판정·검산표가
#   같은 갈래여야 한다(R38 ⑦)
if not NUL_OK:
    run.log("  ⛔ **판정 보류 — 대비의 영점을 측정하지 못했다.**")
    run.log(f"     매크로 Δ {E2['mean']:+.4f} · 전역 Δ {E3['mean']:+.4f} 는 **읽지 않는다**.")
    run.log(f"     문턱을 0 으로 되돌리는 건 「측정」이 아니라 「가정」이다(R26). "
            f"N_PERM({N_PERM})·레코드 수를 늘려 다시 돌려야 한다")
elif ok_("E2") and ok_("E3"):
    run.log("  ★★★ **방법 B 가 A 를 두 지표 모두에서 이긴다.**")
    run.log(f"     매크로 {E2['mean']:+.4f} (문턱 {E2_THR:+.4f}) · "
            f"전역 {E3['mean']:+.4f} (문턱 {E3_THR:+.4f})")
    run.log("     → 층② 의 처방은 **B(burden 을 특징으로)** 이고, 남은 건 `π̂` 를 고치는 것이다")
    run.log(f"     → 배포판 실측 — 매크로 {E4m['mean']:+.4f} · 전역 {E4p['mean']:+.4f}")
elif ok_("E2"):
    run.log("  ★★ **B 는 레코드 내 판별을 개선한다 — A 가 원리적으로 못 하는 일이다.**")
    run.log(f"     매크로 {E2['mean']:+.4f} [{E2['lo']:+.4f}, {E2['hi']:+.4f}] (영점 상단 {E2_THR:+.4f})")
    run.log(f"     전역은 {VERD['E3']} — {E3['mean']:+.4f} [{E3['lo']:+.4f}, {E3['hi']:+.4f}]")
    run.log("     → **R11 이 원래 요구한 지표에서 B 가 이긴다**는 게 이 갈래의 결론이다.")
    run.log("       전역은 지배 지분이 커서(위 R11 경고) 단독으로 결론을 못 낸다")
elif no_("E2"):
    run.log("  ⛔ **B 가 A 에 못 미친다** — 매크로 CI 상단이 문턱 아래다.")
    run.log(f"     매크로 {E2['mean']:+.4f} [{E2['lo']:+.4f}, {E2['hi']:+.4f}] vs 문턱 {E2_THR:+.4f}")
    run.log(f"     전역 {E3['mean']:+.4f} [{E3['lo']:+.4f}, {E3['hi']:+.4f}] vs 문턱 {E3_THR:+.4f}")
    run.log("     → 층② 의 처방은 **A 계열**이고 병목은 무라벨 사전확률 추정이다")
else:
    run.log("  ⚠️ **미결 — 가르지 못했다.** 「B 가 A 와 등가」가 **아니다**(R29 ① · R33 ①).")
    run.log(f"     매크로 {E2['mean']:+.4f} [{E2['lo']:+.4f}, {E2['hi']:+.4f}] · "
            f"MDE {E2['mde']:.4f} vs 문턱 {E2_THR:+.4f}")
    run.log(f"     전역 {E3['mean']:+.4f} [{E3['lo']:+.4f}, {E3['hi']:+.4f}] · "
            f"MDE {E3['mde']:.4f} vs 문턱 {E3_THR:+.4f}")
    run.log(f"     → 필요표본(레코드) — 매크로 {NEED['E2 매크로']['sup80']:.0f} · "
            f"전역 {NEED['E3 전역']['sup80']:.0f} (80% · 현재 {len(REC_OK)}). "
            "효과가 영점에 붙어 있으면 이 수는 해석 불가다(R41 ②)")
run.log("")
run.log("  ▸ ★ 1차의 전역 수치는 **단일 분할(TEST 20)** 의 것이라 직접 비교하지 않았다")
run.log("  ▸ ★ 주 관문을 바꾼 이유 — 사전등록 문구가 「별도 특징」인데 1차가 그걸 밀어냈다")
run.log(f"  ▸ ★ **전역 단독 인용 금지** — 지배 지분 {DOMINANT:.3f}(R11)")

run.finish({
    "exp_id": "quest46_q4b_burden_feature_v2",
    "metric": "b_add_minus_a_macro",
    "value": float(E2["mean"]),
    "passed": bool(ok_("E0") and ok_("E2")),
    "summary": ("Q4 주 관문 교체 — 1차는 「상수 특징 = A 와 구조 동일」이라는 반증된 이론으로 "
                "`B_add` 를 주 관문에서 뺐고 가장 잘한 팔이 밖에 있었다. 주 관문을 `B_add` 대 A 로 "
                "되돌리고, 매크로를 공동 주 지표로 승격(A 가 원리적으로 못 움직이는 지표), "
                "`B_add_em` 배포판을 추가하고, LORO 로 56 레코드 전부를 평가에 썼다."),
    "verdicts": VERD, "rule_check": RULE_CHECK, "cohort": CONFIG.get("cohort", {}),
    "E0": CONFIG.get("E0", {}), "E1": CONFIG.get("E1", {}), "E2": CONFIG.get("E2", {}),
    "E3": CONFIG.get("E3", {}), "E4": CONFIG.get("E4", {}), "E5": CONFIG.get("E5", {}),
    "E6": CONFIG.get("E6", {}), "null_diff": CONFIG.get("null_diff", {}),
    "need": CONFIG.get("need", {}), "E7": CONFIG.get("E7", []), "fig": PNG})
run.log(f"\n저장 완료 — {run.dir}")
run.log("다음: `python pipelines/ingest_run.py --results result.json "
        "--notebook notebooks/quest46_q4b_burden_feature_v2.ipynb`")
